# 01 - GloVe Embeddings + Cosine Similarity

In this notebook, I test sentence similarity with **pre-trained GloVe word embeddings**. I use the 100-dimensional and 300-dimensional versions of the 2024 Wikipedia + Gigaword model:

- [GloVe 2024 Wikipedia + Gigaword, 100d](https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.100d.zip)
- [GloVe 2024 Wikipedia + Gigaword, 300d](https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.300d.zip)

GloVe represents each word as a dense numerical vector learned from a large text collection. Words used in similar situations should have vectors that are close together.

I compare three ways of producing a similarity score:

1. **Mean Pooling Sentence Embeddings:** average all word vectors in each sentence, then compare the two sentence vectors.
2. **One-Way Word Alignment:** for every word in sentence 1, find its most similar word in sentence 2.
3. **Bidirectional Word Alignment:** calculate alignment in both directions and average the two scores.

GloVe can recognise some semantic relationships that TF-IDF cannot. However, it gives each word one fixed vector and does not properly understand word order or the context of a sentence.

In [37]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from nltk.tokenize import word_tokenize

In [38]:
dataset = load_dataset("sentence-transformers/stsb")
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1379
    })
})

## 1. Load Pre-trained GloVe Embeddings

In [39]:
glove100_path = "../../../datasets/glove/wiki_giga_2024_100.txt"
glove300_path = "../../../datasets/glove/wiki_giga_2024_300.txt"

I store in `vocab` purely alphabetic tokens in STS-B Dataset splits and convert them to lowercase, because the selected GloVe files are uncased, to create a vocabulary from GloVe Embeddings.

In [40]:
vocab = set()

for split in ["train", "validation", "test"]:
    for sample in dataset[split]:

        vocab.update(token.lower() for token in word_tokenize(sample["sentence1"]) if token.isalpha())
        vocab.update(token.lower() for token in word_tokenize(sample["sentence2"]) if token.isalpha())


In [41]:
len(vocab)

13574

The dataset contains **13,574 unique alphabetic tokens**.

In [42]:
with open(glove100_path, "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline())

the 0.306717 -0.32053 -0.39364699999999997 0.08282600000000001 0.073522 -0.409154 -0.265564 -0.23693999999999998 -0.305832 0.74529 0.214341 0.27678099999999994 -0.152797 -0.127524 0.11952500000000002 0.640965 -0.175869 0.160711 0.47797799999999996 -0.160939 -0.150093 0.674601 -0.099565 0.021881999999999985 -0.032770999999999995 0.368641 -0.08701900000000007 -0.13332599999999997 0.170143 0.15693399999999996 0.6775059999999999 -0.099686 0.392113 0.37343400000000004 -5.736062 0.413845 0.477368 -0.04169700000000001 0.38310900000000003 0.12015199999999998 -0.20947 0.605104 0.23635299999999998 0.15113100000000002 -0.508865 0.671239 -0.300263 -0.267927 2.549487 0.06717699999999999 0.217224 -0.031316 0.05231 0.119321 -0.332154 -0.8079040000000001 -0.546453 -0.04439199999999999 -0.281657 0.286647 0.32577500000000004 -0.021960000000000007 -0.636903 -0.268063 0.247956 -0.402493 0.276707 -0.275139 0.20115899999999998 0.08284399999999997 0.591695 -0.017126999999999948 -0.09226899999999999 0.3920079

In [76]:
# Store Glove 100d vector Embeddings, for dataset vocabulary
glove100_embeddings = {}

# Read the embeddings
with open(glove100_path, "r", encoding="utf-8") as file:
    for line in file:
        values = line.strip().split()
        word = values[0] # extract the word

        # Check if word in dataset vocabulary
        if word not in vocab:
            continue
      
        vector = np.asarray(values[1:], dtype=np.float32)

        if len(vector) == 100:
            glove100_embeddings[word] = vector


ValueError: could not convert string to float: 'rediff.com'

Some embeddings contain `rediff.com` in embedding vector, I will solve this error by `try/except` block.

In [75]:
# Store Glove 100d vector Embeddings, for dataset vocabulary
glove100_embeddings = {}

# Read the embeddings
with open(glove100_path, "r", encoding="utf-8") as file:
    for line in file:
        values = line.strip().split()
        word = values[0] # extract the word

        # Check if word in dataset vocabulary
        if word not in vocab:
            continue

        # Handle with `ValueError`
        try:
            vector = np.asarray(values[1:], dtype=np.float32)

            if len(vector) == 100:
                glove100_embeddings[word] = vector

        except ValueError:
            continue

In [ ]:
glove100_embeddings["the"]

array([ 0.306717, -0.32053 , -0.393647,  0.082826,  0.073522, -0.409154,
       -0.265564, -0.23694 , -0.305832,  0.74529 ,  0.214341,  0.276781,
       -0.152797, -0.127524,  0.119525,  0.640965, -0.175869,  0.160711,
        0.477978, -0.160939, -0.150093,  0.674601, -0.099565,  0.021882,
       -0.032771,  0.368641, -0.087019, -0.133326,  0.170143,  0.156934,
        0.677506, -0.099686,  0.392113,  0.373434, -5.736062,  0.413845,
        0.477368, -0.041697,  0.383109,  0.120152, -0.20947 ,  0.605104,
        0.236353,  0.151131, -0.508865,  0.671239, -0.300263, -0.267927,
        2.549487,  0.067177,  0.217224, -0.031316,  0.05231 ,  0.119321,
       -0.332154, -0.807904, -0.546453, -0.044392, -0.281657,  0.286647,
        0.325775, -0.02196 , -0.636903, -0.268063,  0.247956, -0.402493,
        0.276707, -0.275139,  0.201159,  0.082844,  0.591695, -0.017127,
       -0.092269,  0.392008,  0.078245, -0.049907,  0.235151,  0.457376,
       -0.111987, -0.05691 ,  0.065092,  0.106512, 

In [ ]:
glove300_embeddings = {}

with open(glove300_path, "r", encoding="utf-8") as file:
    for line in file:
        values = line.strip().split()
        word = values[0]

        # Check if word in dataset vocabulary
        if word not in vocab:
            continue

        try:
            vector = np.asarray(values[1:], dtype=np.float32)

            if len(vector) == 300:
                glove300_embeddings[word] = vector

        except ValueError:
            continue

In [ ]:
glove300_embeddings["the"]

array([-1.527700e-01, -9.299900e-02, -2.290520e-01, -4.476380e-01,
        3.631780e-01,  1.062330e-01, -1.394820e-01, -1.042690e-01,
       -2.833000e-03,  5.523600e-02,  1.800980e-01,  1.390300e-02,
       -4.246300e-02, -2.437670e-01,  2.960380e-01, -3.458110e-01,
        1.675960e-01,  4.835670e-01, -1.356860e-01, -9.374700e-02,
       -4.394700e-02, -1.866110e-01,  1.423750e-01,  8.977800e-02,
       -2.789240e-01,  1.593250e-01, -2.218400e-02, -5.068430e-01,
        1.213680e-01,  1.659180e-01, -4.431800e-02,  6.265100e-02,
       -1.758110e-01, -2.244690e-01, -1.500460e-01, -3.312880e-01,
        1.353720e-01,  5.387800e-02,  1.146700e-02, -1.401600e-01,
        4.844000e-03,  2.154030e-01, -6.709090e-01,  3.944710e-01,
        1.725280e-01,  2.344070e-01, -5.119800e-02, -2.676300e-02,
       -1.546300e-02,  2.238580e-01, -3.865250e-01,  1.443290e-01,
       -4.273360e-01,  3.425000e-02, -5.607800e-02,  2.291290e-01,
       -1.042290e-01,  5.115600e-02,  7.119200e-02,  2.397310e

## 2. Mean Pooling Approach

Mean pooling creates one sentence vector by averaging all known word vectors in that sentence. 

It is simple and fast, but common words can strongly affect the average. It also loses information about which words match each other.

In [77]:
# Define this function to convert sentences tokens to embeddings
def sentence_to_glove_embeddings(sentence, model):
    
    tokens = [token.lower() for token in word_tokenize(sentence) if token.isalpha()]

    word_vectors = []

    # Extract token embeddings from model
    for token in tokens:
        if token in model:
            word_vectors.append(model[token])

    return np.asarray(word_vectors)

In [78]:
def mean_pooling_similarity(emb1, emb2):
      
    # Apply mean pooling to each matrix, then reshape the 1D vectors to 2D arrays (1, 100)
    mean_emb1 = np.mean(emb1, axis=0).reshape(1, -1)
    mean_emb2 = np.mean(emb2, axis=0).reshape(1, -1)
    
    # Compute cosine similarity score
    similarity = cosine_similarity(mean_emb1, mean_emb2)[0][0]
    
    return similarity

In [79]:
glove_mean_pooling = []

for sample in dataset["validation"]:

    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    emb1 = sentence_to_glove_embeddings(sent1, glove100_embeddings)
    emb2 = sentence_to_glove_embeddings(sent2, glove100_embeddings)

    similarity = mean_pooling_similarity(emb1, emb2)

    glove_mean_pooling.append({"sentence1": sent1,
                             "sentence2": sent2,
                             "human_score": score,
                             "pred_score": similarity})

In [80]:
glove_mean_pooling_df = pd.DataFrame(glove_mean_pooling)
glove_mean_pooling_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.987303
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.993743
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,0.983530
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.983570
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.986273


In [81]:
mean_pooling_pearson, _ = pearsonr(glove_mean_pooling_df["pred_score"], glove_mean_pooling_df["human_score"])
mean_pooling_spearman, _ = spearmanr(glove_mean_pooling_df["pred_score"], glove_mean_pooling_df["human_score"])

print(f"GloVe Embeddings Mean Pooling Approach Pearson: {mean_pooling_pearson:.4f}")
print(f"GloVe Embeddings Mean Pooling Approach Spearman: {mean_pooling_spearman:.4f}")

GloVe Embeddings Mean Pooling Approach Pearson: 0.4511
GloVe Embeddings Mean Pooling Approach Spearman: 0.5329


The mean pooling method reaches `0.4511 Pearson` and `0.5329 Spearman` correlation. Spearman is higher by `0.0818`, which means the method is better at putting sentence pairs in roughly the correct order than predicting the exact size of the human scores.

The example rows also show a weakness of mean pooling. 

`"A woman is playing the guitar"` and `"A man is playing guitar"` receive a very high prediction of `0.9836`, while the human score is only `0.48`. 

The shared words dominate the average, so the change from *woman* to *man* has too little effect.

## 2. One-Way Word Alignment

For each word in sentence 1, one-way alignment finds the most similar word in sentence 2. The final score is the average of these best matches. 

This keeps more token level information than mean pooling.

In [50]:
def one_way_alignment(emb1, emb2):
    
    # Get cosine similarity matrix
    cos_sim = cosine_similarity(emb1, emb2)
    
    # One way max-pooling alignment
    one_way_sim = np.mean(np.max(cos_sim, axis=1))
    
    return one_way_sim 

In [51]:
glove_one_way = []

for sample in dataset["validation"]:

    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    emb1 = sentence_to_glove_embeddings(sent1, glove100_embeddings)
    emb2 = sentence_to_glove_embeddings(sent2, glove100_embeddings)

    similarity = one_way_alignment(emb1, emb2)

    glove_one_way.append({"sentence1": sent1,
                          "sentence2": sent2,
                          "human_score": score,
                          "pred_score": similarity})

In [52]:
glove_one_way_df = pd.DataFrame(glove_one_way)
glove_one_way_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.980660
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.965336
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,1.000000
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.940642
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.940642


In [53]:
glove_one_way_pearson, _ = pearsonr(glove_one_way_df["pred_score"], glove_one_way_df["human_score"])
glove_one_way_spearman, _ = spearmanr(glove_one_way_df["pred_score"], glove_one_way_df["human_score"])

print(f"GloVe Embeddings One-Way Directional Pearson: {glove_one_way_pearson:.4f}")
print(f"GloVe Embeddings One-Way Directional Spearman: {glove_one_way_spearman:.4f}")

GloVe Embeddings One-Way Directional Pearson: 0.6105
GloVe Embeddings One-Way Directional Spearman: 0.6292


One-way word alignment improves **Pearson** correlation from `0.4511 to 0.6105`, an increase of `0.1594`. 

**Spearman** correlation rises from `0.5329 to 0.6292`, an increase of `0.0963`. This suggests that matching individual words works much better than representing each full sentence with only one average vector.

However, this method is directional. Extra information in sentence 2 may not reduce the score enough because only words from sentence 1 must find a match.

## 3. Bidirectional Alignment

Bidirectional alignment solves the direction problem by calculating the best word matches from sentence 1 to sentence 2 and also from sentence 2 to sentence 1. The two directional scores are then averaged.

In [ ]:
def bidirectional_alignment(emb1, emb2):

    cos_sim = cosine_similarity(emb1, emb2)
    
    # Bidirectional max-pooling alignment
    sim_s1_s2 = np.mean(np.max(cos_sim, axis=1))
    sim_s2_s1 = np.mean(np.max(cos_sim, axis=0))
    
    return (sim_s1_s2 + sim_s2_s1) / 2

In [ ]:
glove_bidirectional = []

for sample in dataset["validation"]:

    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    emb1 = sentence_to_glove_embeddings(sent1, glove100_embeddings)
    emb2 = sentence_to_glove_embeddings(sent2, glove100_embeddings)

    similarity = bidirectional_alignment(emb1, emb2)

    glove_bidirectional.append({"sentence1": sent1,
                                "sentence2": sent2,
                                "human_score": score,
                                "pred_score": similarity})

In [58]:
glove_bidirectional_df = pd.DataFrame(glove_bidirectional)
glove_bidirectional_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.970462
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.982668
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,0.979978
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.952726
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.955658


In [59]:
glove_bidirectional_pearson, _ = pearsonr(glove_bidirectional_df["pred_score"], glove_bidirectional_df["human_score"])
glove_bidirectional_spearman, _ = spearmanr(glove_bidirectional_df["pred_score"], glove_bidirectional_df["human_score"])

print(f"GloVe Embeddings Bi-Directional Pearson: {glove_bidirectional_pearson:.4f}")
print(f"GloVe Embeddings Bi-Directional Spearman: {glove_bidirectional_spearman:.4f}")

GloVe Embeddings Bi-Directional Pearson: 0.6397
GloVe Embeddings Bi-Directional Spearman: 0.6567


With 100-dimensional GloVe vectors, bidirectional alignment reaches `0.6397 Pearson` and `0.6567 Spearman`. 

Compared with one-way alignment, Pearson improves by `0.0292` and Spearman improves by `0.0275`. The increase is not very large, but it is consistent for both metrics.

## 4. Bidirectional Approach with GloVe 300d Embedding

Now, I repeat the strongest validation method, bidirectional alignment, with the larger 300-dimensional GloVe vectors. 

More dimensions can store more information about each word, but they also require more memory and computation.

In [60]:
glove300_bidirectional = []

for sample in dataset["validation"]:

    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    emb1 = sentence_to_glove_embeddings(sent1, glove300_embeddings)
    emb2 = sentence_to_glove_embeddings(sent2, glove300_embeddings)

    similarity = bidirectional_alignment(emb1, emb2)

    glove300_bidirectional.append({"sentence1": sent1,
                                   "sentence2": sent2,
                                   "human_score": score,
                                   "pred_score": similarity})

In [61]:
glove300_bidirectional_df = pd.DataFrame(glove300_bidirectional)
glove300_bidirectional_df.head()

,sentence1,sentence2,human_score,pred_score
0,A man with a hard hat is dancing.,A man wearing a hard hat is dancing.,1.00,0.953214
1,A young child is riding a horse.,A child is riding a horse.,0.95,0.973870
2,A man is feeding a mouse to a snake.,The man is feeding a mouse to the snake.,1.00,0.964285
3,A woman is playing the guitar.,A man is playing guitar.,0.48,0.925981
4,A woman is playing the flute.,A man is playing a flute.,0.55,0.930275


In [69]:
glove300_bidirectional_pearson, _ = pearsonr(glove300_bidirectional_df["pred_score"], glove300_bidirectional_df["human_score"])
glove300_bidirectional_spearman, _ = spearmanr(glove300_bidirectional_df["pred_score"], glove300_bidirectional_df["human_score"])

print(f"GloVe 300d Embeddings Bi-Directional Pearson: {glove300_bidirectional_pearson:.4f}")
print(f"GloVe 300d Embeddings Bi-Directional Spearman: {glove300_bidirectional_spearman:.4f}")

GloVe 300d Embeddings Bi-Directional Pearson: 0.6574
GloVe 300d Embeddings Bi-Directional Spearman: 0.6652


The 300d model gives the best validation result: `0.6574 Pearson` and `0.6652 Spearman`. 

Compared with the 100d bidirectional model, Pearson increases by `0.0177` and Spearman increases by `0.0085`. 

The improvement is real but small, so the 100d model may still be useful when memory or speed is more important.

## 5. Test Set Evaluation

After selecting 300d bidirectional alignment as the best method on the validation set, I evaluate it once on the test set. 

This gives a better estimate of how well the selected method works on unseen sentence pairs.

In [66]:
test_glove300_bidirectional = []

for sample in dataset["test"]:

    sent1 = sample["sentence1"]
    sent2 = sample["sentence2"]
    score = sample["score"]

    emb1 = sentence_to_glove_embeddings(sent1, glove300_embeddings)
    emb2 = sentence_to_glove_embeddings(sent2, glove300_embeddings)

    similarity = bidirectional_alignment(emb1, emb2)

    test_glove300_bidirectional.append({"sentence1": sent1,
                                   "sentence2": sent2,
                                   "human_score": score,
                                   "pred_score": similarity})

In [67]:
test_glove_df = pd.DataFrame(test_glove300_bidirectional)
test_glove_df.head()

,sentence1,sentence2,human_score,pred_score
0,A girl is styling her hair.,A girl is brushing her hair.,0.50,0.894849
1,A group of men play soccer on the beach.,A group of boys are playing soccer on the beach.,0.72,0.925529
2,One woman is measuring another woman's ankle.,A woman measures another woman's ankle.,1.00,0.874699
3,A man is cutting up a cucumber.,A man is slicing a cucumber.,0.84,0.906335
4,A man is playing a harp.,A man is playing a keyboard.,0.30,0.891220


In [70]:
glove_test_pearson, _ = pearsonr(test_glove_df["pred_score"], test_glove_df["human_score"])
glove_test_spearman, _ = spearmanr(test_glove_df["pred_score"], test_glove_df["human_score"])

print(f"GloVe 300d Embeddings Bi-Directional Test Set, Pearson: {glove_test_pearson:.4f}")
print(f"GloVe 300d Embeddings Bi-Directional Test Set, Spearman: {glove_test_spearman:.4f}")

GloVe 300d Embeddings Bi-Directional Test Set, Pearson: 0.5382
GloVe 300d Embeddings Bi-Directional Test Set, Spearman: 0.5287


## 6. Result Comparison and Conclusion

In [74]:
print(f"GloVe Embeddings Mean Pooling Approach Pearson: {mean_pooling_pearson:.4f}")
print(f"GloVe Embeddings Mean Pooling Approach Spearman: {mean_pooling_spearman:.4f}")
print("--" * 40)
print(f"GloVe Embeddings One-Way Directional Pearson: {glove_one_way_pearson:.4f}")
print(f"GloVe Embeddings One-Way Directional Spearman: {glove_one_way_spearman:.4f}")
print("--" * 40)
print(f"GloVe Embeddings Bi-Directional Pearson: {glove_bidirectional_pearson:.4f}")
print(f"GloVe Embeddings Bi-Directional Spearman: {glove_bidirectional_spearman:.4f}")
print("--" * 40)
print(f"GloVe 300d Embeddings Bi-Directional Pearson: {glove300_bidirectional_pearson:.4f}")
print(f"GloVe 300d Embeddings Bi-Directional Spearman: {glove300_bidirectional_spearman:.4f}")
print("--" * 40)
print(f"GloVe 300d Embeddings Bi-Directional Test Set, Pearson: {glove_test_pearson:.4f}")
print(f"GloVe 300d Embeddings Bi-Directional Test Set, Spearman: {glove_test_spearman:.4f}")

GloVe Embeddings Mean Pooling Approach Pearson: 0.4511
GloVe Embeddings Mean Pooling Approach Spearman: 0.5329
--------------------------------------------------------------------------------
GloVe Embeddings One-Way Directional Pearson: 0.6105
GloVe Embeddings One-Way Directional Spearman: 0.6292
--------------------------------------------------------------------------------
GloVe Embeddings Bi-Directional Pearson: 0.6397
GloVe Embeddings Bi-Directional Spearman: 0.6567
--------------------------------------------------------------------------------
GloVe 300d Embeddings Bi-Directional Pearson: 0.6574
GloVe 300d Embeddings Bi-Directional Spearman: 0.6652
--------------------------------------------------------------------------------
GloVe 300d Embeddings Bi-Directional Test Set, Pearson: 0.5382
GloVe 300d Embeddings Bi-Directional Test Set, Spearman: 0.5287


The results show that **the similarity method** makes a bigger difference than **increasing vector size**. Moving from mean pooling to 100d bidirectional alignment improves Pearson by `0.1886`, while moving from 100d to 300d bidirectional alignment improves it by only `0.0177`.

**The final test result is lower than the best validation result.** Pearson falls from `0.6574 to 0.5382`, a difference of `0.1192`, and Spearman falls from `0.6652 to 0.5287`, a difference of `0.1365`. This shows that the method does **not generalise well** to every group of sentence pairs.

**The predicted scores are also often very high, even when the human score is moderate.** 

`"A man is playing a harp"` and `"A man is playing a keyboard"` receive `0.8912`, while the human score is `0.30`. 

GloVe knows that `harp` and `keyboard` are musical instruments, but it does not fully understand that they describe different actions in this task.

Overall, 300d bidirectional alignment is the strongest GloVe approach tested here. It benefits from token level matching and richer vectors, but it still has limits because GloVe uses fixed word meanings and does not understand sentence context or word order.